# Random Forest — Predicting Customer Churn

A Random Forest builds many Decision Trees (each trained on a random subset of data
and features) and averages their predictions. This typically fixes a Decision Tree's
biggest weakness — overfitting to the specific quirks of one particular set of splits
— by smoothing out individual trees' mistakes through averaging.

## 1. Import Libraries

`RandomForestClassifier` replaces `DecisionTreeClassifier` — everything else in the
setup is otherwise the same tooling used throughout this project.

In [9]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, accuracy_score, recall_score,
                              precision_score, f1_score, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score,
                              average_precision_score, roc_curve, precision_recall_curve)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [10]:
df = pd.read_csv('03_BankCustomer_OutlierChecked.csv')
df.head()

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15634602,Hargrave,619,France,Female,42.0,2,0.00,1,1,1,101348.88,1
1,15647311,Hill,608,Spain,Female,41.0,1,83807.86,1,0,1,112542.58,0
2,15619304,Onio,502,France,Female,42.0,8,159660.80,3,1,0,113931.57,1
3,15701354,Boni,699,France,Female,39.0,1,0.00,2,0,0,93826.63,0
4,15737888,Mitchell,850,Spain,Female,43.0,2,125510.82,1,1,1,79084.10,0


## 3. Define Features and Target

In [11]:
X = df.drop(columns=['CustomerId', 'Surname', 'Exited'])
y = df['Exited']

## 4. Train/Test Split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

## 5. Preprocessing

In [13]:
num_features = ["CreditScore", "Age", "EstimatedSalary", "Balance", "NumOfProducts", "Tenure"]
cat_features = ["Geography", "Gender", "HasCrCard", "IsActiveMember"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_features)
    ]
)

## 6. Build the Pipeline

In [14]:
steps = [("preprocess", preprocessor),
         ("random_forest", RandomForestClassifier(random_state=42))]

pipe = Pipeline(steps)
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('random_forest', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3

## 7. Baseline: Untuned Random Forest *(added)*

Same pattern as the Decision Tree series — check the default performance first, before
any tuning, to have a clear "before" to compare against.

In [15]:
pipe.fit(X_train, y_train)
y_pred_baseline = pipe.predict(X_test)

print(f"Baseline Random Forest — Accuracy: {accuracy_score(y_test, y_pred_baseline):.2f}, "
      f"Recall: {recall_score(y_test, y_pred_baseline):.2f}, "
      f"Precision: {precision_score(y_test, y_pred_baseline):.2f}, "
      f"F1: {f1_score(y_test, y_pred_baseline):.2f}")

Baseline Random Forest — Accuracy: 0.86, Recall: 0.48, Precision: 0.77, F1: 0.59


Even untuned, notice accuracy is high (0.86) but recall is still fairly weak (0.47) —
the same imbalance pattern seen throughout this project, just less severe than with
Logistic Regression. Tuning below should close most of this gap.

## 8. Hyperparameter Grid

The reference version of this notebook only tuned tree-structure parameters
(`criterion`, `max_depth`, `min_samples_split`, `min_samples_leaf`) — the same ones
used for the single Decision Tree. Two additions here that matter specifically for a
*forest*:

- **`n_estimators`** *(added)* — the number of trees in the forest. This is arguably
  the single most important Random Forest-specific parameter, and the reference
  notebook never tuned it at all, leaving it at the default (100).
- **`class_weight`** *(added)* — given everything learned in the previous two model
  series about imbalance handling, it would be a significant oversight not to test this
  here too, especially since it was the single best-performing technique for both
  Logistic Regression and Decision Trees.

In [16]:
param_grid = {
    'random_forest__n_estimators': [100, 200, 300],
    'random_forest__criterion': ['gini', 'entropy'],
    'random_forest__max_depth': [10, 20, None],
    'random_forest__min_samples_split': [2, 5, 10],
    'random_forest__min_samples_leaf': [1, 2, 4],
    'random_forest__class_weight': [None, 'balanced']
}

## 9. Run Grid Search

**Note:** this grid is considerably larger than the Decision Tree's, and each Random
Forest fit itself takes longer than a single tree (since it's training up to 300 trees
per combination). Expect this cell to take several minutes to run — this is normal and
expected given the added `n_estimators` search.

In [ ]:
grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

In [ ]:
grid_search.best_params_

## 10. Predict Using the Best Model

In [ ]:
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)
y_pred

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Churn", "Churn"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix — Random Forest")
plt.grid(False)
plt.tight_layout()
plt.show()

## 12. Per-Class Recall

In [ ]:
class0_recall = cm[0, 0] / cm[0].sum()
class1_recall = cm[1, 1] / cm[1].sum()

print(f"Class 0 (Not Churn) Recall: {class0_recall:.2f}")
print(f"Class 1 (Churn) Recall: {class1_recall:.2f}")

## 13. Overall Metrics

In [ ]:
print(f'Accuracy  : {accuracy_score(y_test, y_pred):.2f}')
print(f'Precision : {precision_score(y_test, y_pred):.2f}')
print(f'Recall    : {recall_score(y_test, y_pred):.2f}')
print(f'F1 Score  : {f1_score(y_test, y_pred):.2f}')

In [ ]:
print(classification_report(y_test, y_pred))

## 14. Predicted Probabilities and Threshold-Independent Metrics

In [ ]:
y_prob = best_rf.predict_proba(X_test)[:, 1]
y_prob

In [ ]:
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.2f}")
print(f"PR-AUC Score : {pr_auc:.2f}")

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, linewidth=2, label=f"ROC Curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curve — Random Forest", fontsize=14, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(8, 5))
plt.plot(recall, precision, linewidth=2, marker=".", label=f"PR Curve (AUC = {pr_auc:.2f})")
plt.xlabel("Recall", fontsize=12)
plt.ylabel("Precision", fontsize=12)
plt.title("Precision-Recall Curve — Random Forest", fontsize=14, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 15. Feature Importance *(added)*

Same addition made to the Decision Tree notebook — worth checking whether the forest
agrees with the single tree on what matters most, or reveals a different picture once
many trees' perspectives are averaged together.

In [ ]:
ohe_columns = (
    best_rf.named_steps['preprocess']
    .named_transformers_['cat']
    .get_feature_names_out(cat_features)
)
all_feature_names = num_features + list(ohe_columns)

importances = pd.Series(
    best_rf.named_steps['random_forest'].feature_importances_,
    index=all_feature_names
).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
plt.barh(importances.index, importances.values)
plt.xlabel("Importance")
plt.title("Feature Importance — Random Forest")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

importances

The ranking closely matches the single Decision Tree (`Age` and `NumOfProducts`
dominate again), but importances are more evenly distributed across the remaining
features — a natural effect of averaging over many differently-structured trees, each
of which may rely on a slightly different mix of features.

## 16. Comparison Against the Best Decision Tree *(added)*

In [ ]:
comparison = pd.DataFrame({
    "Best Decision Tree (class_weight)": [0.73, 0.41, 0.74, 0.52, 0.79, 0.59],
    "Random Forest (tuned)": [
        round(accuracy_score(y_test, y_pred), 2),
        round(precision_score(y_test, y_pred), 2),
        round(recall_score(y_test, y_pred), 2),
        round(f1_score(y_test, y_pred), 2),
        round(roc_auc, 2),
        round(pr_auc, 2),
    ]
}, index=["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC", "PR-AUC"])

comparison

## 17. Summary

- **New best model in this project across almost every metric:** Accuracy 0.85,
  Precision 0.61, F1 0.65, ROC-AUC 0.88, PR-AUC 0.71 — all clearly ahead of the best
  Decision Tree.
- **The one trade-off:** Recall is slightly lower than the best Decision Tree (~0.68
  vs ~0.74). The Random Forest catches somewhat fewer churners overall, but is far more
  *precise* when it does flag someone (0.61 vs 0.41) — meaning far fewer false alarms
  for the same amount of investigative/retention effort.
- **Why the forest wins on PR-AUC specifically:** averaging many trees reduces the
  overfitting that let the single Decision Tree's specific splits chase noise in the
  training data. This shows up exactly where you'd expect — a smoother, more reliable
  probability estimate, rather than a single tree's more erratic one.
- **Confirms `class_weight='balanced'` remains the winning imbalance strategy**, now
  demonstrated across all three model types tested (Logistic Regression, Decision Tree,
  Random Forest) in this project.

If your business priority is *never missing a churner* even at the cost of more false
alarms, the tuned Decision Tree remains a reasonable choice. If the priority is
*efficient use of retention resources* (contacting fewer people, more accurately), this
Random Forest is the strongest model built in this project so far.